In [29]:
from functools import cache
from itertools import product
import numpy as np
import random
import cProfile
import pstats
import copy

K.<ω> = CyclotomicField(3)       
O = K.ring_of_integers()     

@cache 
def cubic_residue(a, π):
    a = O(a)
    red, w, e, one = residue_map(π)
    if red(a) == 0:
        return 0
    r = red(a)**e
    if r == one:
        return 0
    elif r == w:
        return 1
    else:
        return 2
@cache
def residue_map(π):
    π = O(π)
    P = O.fractional_ideal(π)
    k = K.residue_field(P) 
    red = k.reduction_map()
    w = red(ω)
    e = (π.norm() - 1) // 3
    one = k(1)
    return red, w, e, one

"""
Returns the base ω log of the general cubic reciprocity (a/b)
"""
@cache
def cr(a,b):
    res = 0
    for π, exp in prime_factors(b):
        local = cubic_residue(a, π)
        res += local*exp 
    return res % 3
    
"""
Returns the prime factorization of b in the ring O
"""
@cache
def prime_factors(b):
    return O(b).factor()

"""
Returns the prime above a 1 mod 3 prime in the ring O
"""
@cache
def prime_above(p):
    # p needs to be a prime 1 mod 3 here
    fac = prime_factors(p)
    for π, e in fac:
        a, b = π.polynomial().coefficients(sparse=False)[::-1]
        if b >= 0:
            return π
    return fac[0][0]

"""
Returns the p-valuation of x
"""
def val(p,x):
    v = 0
    while(x%p==0):
        x/=p
        v+=1
    return v

"""
Returns the prime factorization of b in the integers
"""
@cache
def int_fac(x):
    return Integer(x).factor()

"""
a: int 
b: int
Returns a cleaned (new_a, new_b), such that whenever a prime p divides new_a, p^3 doesn't divide new_b
"""
def clean(a,b):
    new_a = a
    new_b = b
    for p,exp in int_fac(b): 
        if a%p == 0 and exp >= 3:
            v1 = val(p,a)
            v2 = exp
            times = min(v1, v2 // 3)
            new_a /= p ** times
            new_b /= p**(3 * times)
    return new_a, new_b

"""
a: int 
b: list of lists denoting the prime factorization of b
Returns a cleaned (new_a, new_b_fac)
"""
def clean_fac(a, b_fac):
    new_b_fac = []
    new_b = 1
    new_a = a
    for (p, exp) in b_fac:
        if exp < 3 or a % p != 0:
            new_b_fac.append([p, exp])
            new_b *= p ** exp
            continue
        v1 = val(p,a)
        times = min(v1, exp // 3)
        new_a /= p ** times
        new_exp = exp - (3 * times)
        if new_exp > 0:
            new_b_fac.append([p, new_exp])
            new_b *= p ** new_exp
        
    return a, new_b_fac, new_b

In [30]:
"""
As defined in Stephanie Chan's paper
"""
def delta(n):
    r = n %9
    if(r == 3 or r==6):
        return 1
    if (r== 4 or r ==5):
        return -1;
    return 0

"""
As defined in Stephanie Chan's paper
"""
def w2(n):
    v_2 = val(2,n)
    two_n =  2 * n
    if(v_2 == 2):
        two_n /= 8
    cnt = 0
    for p,exp in int_fac(two_n):
        if p%3 == 2:
            cnt+=1
    return cnt

"""
Returns the nullity of an l by m matrix mat
"""
def calc(l, m, mat):
    if l>0 :
        R = matrix(GF(3),np.array(mat))
        return m - R.rank()
    else:
        return m

def build_matrix(primes_to_check, primes_to_use, m, l, t, u1u2u3):
    R = np.empty((l, m), dtype=int) 
    for i in range(l): # l rows for the primes to check
        if i+1<=t:
            for j in range(m): # m columns for the primes to use
                if i==j: 
                    q_i = primes_to_use[i][0] ** primes_to_use[i][1]
                    res = cr(u1u2u3/q_i, prime_above(primes_to_check[i]))  
                    if res == 1: 
                        R[i][j] = 2 
                    elif res == 2: 
                        R[i][j] = 1 
                    else: 
                        R[i][j] = 0
                else:
                    R[i][j] = cr(primes_to_use[j][0]** primes_to_use[i][1], prime_above(primes_to_check[i]))
        else:
            for j in range(m): # m columns 
                R[i][j] = cr(primes_to_use[j][0], prime_above(primes_to_check[i]))
    return l,m,R

def solve(u1u2u3, c):
    dp = 27*(u1u2u3) - c**3
    primes_to_use = []
    t1 = []
    t2 = []
    primes_to_check = []
    for p,exp in Integer(u1u2u3).factor():
        if c%p == 0 and p%3 == 1:
            primes_to_check.append(p)
            t1.append([p,exp])
        else:
            t2.append([p,exp])
    primes_to_use = t1 + t2
        
    m = len(primes_to_use)
    t = len(primes_to_check)
    for p,exp in dp.factor():
        if p%3==1 and (u1u2u3)%p != 0:
            primes_to_check.append(p)
    l = len(primes_to_check)
    return build_matrix(primes_to_check, primes_to_use, m, l, t, u1u2u3)

def solve_fac(u1u2u3, u1u2u3_fac, c):
    dp = 27*(u1u2u3) - c**3
    primes_to_use = []
    t1 = []
    t2 = []
    primes_to_check = []
    for p,exp in u1u2u3_fac:
        if c%p == 0 and p%3 == 1:
            primes_to_check.append(p)
            t1.append([p,exp])
        else:
            t2.append([p,exp])
    primes_to_use = t1 + t2
        
    m = len(primes_to_use)
    t = len(primes_to_check)
    for p,exp in dp.factor():
        if p%3==1 and (u1u2u3)%p != 0:
            primes_to_check.append(p)
    l = len(primes_to_check)
    return build_matrix(primes_to_check, primes_to_use, m, l, t, u1u2u3)

In [33]:
from sympy import nextprime

num_of_primes = 100 # number of primes to use

p1 = []
p2 = []
for i in range(num_of_primes): 
    p = nextprime(3,i+1)
    if p % 3 == 1: 
        p1.append(p)
    else: 
        p2.append(p)

def computeByPrimeFac(num_1_mod_3, num_2_mod_3, N, exponents, H):
    assert num_1_mod_3 +num_2_mod_3 == len(exponents), "the format must match" 
    
    record  = {} # count the number of times each compressed matrix appeared 

    for j in range(N):
        B = 1
        B_fac = []
        primes_to_use = random.sample(range(0, len(p1)), num_1_mod_3)
        for i in range(num_1_mod_3):
            B *= p1[primes_to_use[i]]**int(exponents[i])
            B_fac.append([p1[primes_to_use[i]],int(exponents[i])])
        primes_to_use = random.sample(range(0, len(p2)), num_2_mod_3)
        for i in range(num_2_mod_3):
            B *= p2[primes_to_use[i]]**int(exponents[i+num_1_mod_3 ])
            B_fac.append([p2[primes_to_use[i]],int(exponents[i+num_1_mod_3])])
            
        for A in range(-H, H+1):
            if A%3 ==0: 
                continue
            newA, newB_fac, newB = clean_fac(A,B_fac)
            if A != newA or B != newB: 
                continue
            l,m,mat = solve_fac(newB, newB_fac, newA)
            compressed  = ''.join(str(num) for row in mat for num in row)
            if (l-m, m) not in record: 
                record[(l-m,m)] = {}
            if compressed not in record[(l-m,m)]:
                record[(l-m,m)][compressed] = 1
            else:
                record[(l-m,m)][compressed] += 1 
            # print(newA, newB, l, m, mat)
    return record

In [ ]:
record = computeByPrimeFac(1,2,30,"121",30)